# 00 Ingest GUI Workflow (`adamacs_ingest_v2`)

This is the center-stage ingest workflow notebook for `adamacs_ingest`.

It focuses on:
1. Database connection
2. Loading `adamacs.helpers.adamacs_ingest_v2`
3. Discovering session folders
4. Launching the ingest GUI


## 1) Setup and DB connection

Set `DJ_HOST` and `DJ_USER` in your shell if needed. Password is prompted by the DataJoint connection flow.


In [ ]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

repo_root = Path.cwd()
config_path = repo_root / 'dj_local_conf.json'

import datajoint as dj
if config_path.exists():
    dj.config.load(str(config_path))
else:
    print(f'Warning: config not found at {config_path}; relying on environment variables.')

print(f'Working directory: {repo_root}')
print(f'DataJoint version: {dj.__version__}')
print(f"DB prefix: {dj.config.get('custom', {}).get('database.prefix', '<unset>')}")
dj.conn()


## 2) Load ingest v2 helper

In [ ]:
import adamacs.helpers.adamacs_ingest_v2 as ai
print('Loaded:', ai.__name__)


## 3) Discover session folders

In [ ]:
import fnmatch
import os
from natsort import natsorted
from IPython.display import display, HTML

session_filter = os.environ.get('ADAMACS_SESSION_FILTER', '*')
date_filter = os.environ.get('ADAMACS_DATE_FILTER', '*')

root_dirs = dj.config.get('custom', {}).get('exp_root_data_dir', [])
if not root_dirs:
    raise ValueError('dj.config["custom"]["exp_root_data_dir"] is not configured.')

dataroot = root_dirs[0]
dirs_root = [
    d for d in os.listdir(dataroot)
    if os.path.isdir(os.path.join(dataroot, d))
    and fnmatch.fnmatch(d, session_filter)
    and fnmatch.fnmatch(d, f'*{date_filter}*')
]
sorted_dirs_root = natsorted(dirs_root, reverse=True)

print(f'Data root: {dataroot}')
print(f'Filter: session={session_filter!r}, date={date_filter!r}')
print(f'Found {len(sorted_dirs_root)} candidate sessions.')
for path in sorted_dirs_root:
    display(HTML(f'<a href="{os.path.join(dataroot, path)}" target="_blank">{path}</a>'))


## 4) Launch ingest GUI (opt-in)

Set `ADAMACS_LAUNCH_GUI=1` before running this cell to open the interactive GUI.


In [ ]:
LAUNCH_GUI = os.environ.get('ADAMACS_LAUNCH_GUI', '0') == '1'

if not sorted_dirs_root:
    print('No matching sessions found. Adjust ADAMACS_SESSION_FILTER / ADAMACS_DATE_FILTER and rerun.')
elif not LAUNCH_GUI:
    print('GUI launch skipped. Set ADAMACS_LAUNCH_GUI=1 to open the interactive selector.')
    print('First 10 matching sessions:')
    for path in sorted_dirs_root[:10]:
        print('  -', path)
else:
    print('ADAMACS INGEST GUI v2')
    selected_data, get_dlc_models = ai.select_sessions(
        sorted_dirs_root,
        do_population=False,
        rspace_upload=False,
        ingest_opt='trigger',
    )


## Notes

- Use this notebook as the default ingest entrypoint.
- Keep ad-hoc debugging in separate notebooks/scripts.
